In [ ]:
"""
FM Student Training on teacher pairs
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES
from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 512
EPOCHS = 1001
CKPT_EVERY = 5
LR = 1e-4              # EDM train default; try 1e-5 if unstable
HIDDEN_NF = 128
PAD_TO = MAX_N_NODES
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5   # epochs with < MIN_DELTA improvement
MIN_DELTA = 0.01
PART_CYCLE = [0, 1]       # or [0, 1, 2, 3, 2, 1]

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}.log"


# --------------------- EDM-style helpers ---------------------
class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    def update_model_average(self, ma_model, current_model):
        for cur, ma in zip(current_model.parameters(), ma_model.parameters()):
            ma.data = self.update_average(ma.data, cur.data)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1.0 - self.beta) * new


class Queue:
    def __init__(self, max_len: int = 50):
        self.items = []
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def add(self, item):
        self.items.insert(0, item)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, gradnorm_queue: Queue):
    max_grad_norm = 1.5 * gradnorm_queue.mean() + 2.0 * gradnorm_queue.std()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=max_grad_norm, norm_type=2.0
    )
    gn = float(grad_norm)
    if gn > max_grad_norm:
        gradnorm_queue.add(float(max_grad_norm))
        print(f"Clipped gradient {gn:.1f} (allowed {max_grad_norm:.1f})")
    else:
        gradnorm_queue.add(gn)


def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def load_part_pairs(part_i: int):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt"))
    if not paths:
        # also try part{N}_shard / flat shards
        paths = sorted(PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(f"no part shards; only flat shards in {PAIR_DIR}")
    if not paths:
        raise FileNotFoundError(f"no shards for part {part_i + 1} in {PAIR_DIR}")
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["z_T"] for p in packs]),
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def save_ckpt(path, epoch, avg, best, student, student_ema, opt, gradnorm_queue):
    torch.save(
        {
            "epoch": epoch,
            "avg_loss": avg,
            "best_loss": best,
            "student": student.state_dict(),
            "student_ema": student_ema.state_dict() if student_ema is not None else None,
            "opt": opt.state_dict(),
            "hidden_nf": HIDDEN_NF,
            "lr": LR,
            "batch": BATCH,
            "ema_decay": EMA_DECAY,
            "gradnorm_queue": list(gradnorm_queue.items),
        },
        path,
    )


# --------------------- model / optim ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
    ),
    in_node_nf=8,
).to(device)

student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)

opt = AdamW(
    student.parameters(),
    lr=LR,
    amsgrad=True,
    weight_decay=1e-12,
)

gradnorm_queue = Queue()
gradnorm_queue.add(3000.0)  # same seed value as EDM train

log(
    f"start device={device} batch={BATCH} hidden_nf={HIDDEN_NF} lr={LR} "
    f"ema={EMA_DECAY} epochs={EPOCHS} adaptive_clip=True"
)

# --------------------- train ---------------------
best_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    z_T, x1, n_atoms, context = load_part_pairs(part_i)
    perm = torch.randperm(z_T.shape[0])
    z_T, x1, n_atoms, context = z_T[perm], x1[perm], n_atoms[perm], context[perm]
    log(f"epoch={epoch} part={part_i + 1} n={z_T.shape[0]}")

    student.train()
    running, n_steps = 0.0, 0
    pbar = tqdm(
        range(0, z_T.shape[0] - BATCH + 1, BATCH),
        desc=f"ep{epoch} part{part_i + 1}",
    )
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = n_atoms[sl].to(device=device, dtype=torch.long)
        node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
        ctx = context[sl].to(device=device, dtype=torch.float32)
        normed = (ctx - norms["mean"]) / norms["mad"]
        batch_context = normed.unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask

        x0_b = z_T[sl].to(device=device, dtype=torch.float32)
        x1_b = x1[sl].to(device=device, dtype=torch.float32)

        loss = student.compute_loss(x0_b, x1_b, node_mask, edge_mask, batch_context)
        if not torch.isfinite(loss):
            log(f"WARN non-finite epoch={epoch} step={n_steps}")
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, gradnorm_queue)
        opt.step()
        ema.update_model_average(student_ema, student)

        running += loss.item()
        n_steps += 1
        if n_steps % 20 == 0:
            pbar.set_postfix(loss=f"{running / n_steps:.4f}")

    avg = running / max(n_steps, 1)
    improved = avg < best_loss - MIN_DELTA
    if avg < best_loss:
        best_loss = avg
        save_ckpt(
            CKPT_DIR / f"best_{HIDDEN_NF}.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt best epoch={epoch} avg_loss={avg:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    log(
        f"epoch={epoch} part={part_i + 1} steps={n_steps} "
        f"avg_loss={avg:.6f} best_loss={best_loss:.6f} "
        f"grad_q_mean={gradnorm_queue.mean():.1f}"
    )

    if (epoch + 1) % CKPT_EVERY == 0:
        save_ckpt(
            CKPT_DIR / f"latest_{HIDDEN_NF}.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt latest epoch={epoch} avg_loss={avg:.6f}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        log(f"early stop at epoch={epoch} best_loss={best_loss:.6f}")
        break

log(f"done best_loss={best_loss:.6f}")

/tmp/ipykernel_41441/3913875974.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu")


2026-09-13T08:58:43  start offline device=cuda batch=128 hidden_nf=128 lr=1e-05


/tmp/ipykernel_41441/3913875974.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  z_T = torch.cat([torch.load(p, map_location="cpu")["z_T"] for p in paths])
/tmp/ipykerne

2026-09-13T08:58:44  epoch=0 part=1 n=199680


ep0 part1:  30%|██▉       | 462/1560 [02:48<06:40,  2.74it/s, loss=25.8591]

In [ ]:
"""
Offline FM student: 3 linear segments z_T → z_a → z_b → x1
(z_a = teacher reverse after T/3, z_b after 2T/3).
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 128
EPOCHS = 40
CKPT_EVERY = 5
LR = 1e-4
HIDDEN_NF = 420
PAD_TO = MAX_N_NODES
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5
MIN_DELTA = 0.01
N_SEG = 3
PART_CYCLE = [0, 1]

PAIR_DIR = Path("./teacher_pairs/teacher_pairs_3seg")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_3seg.log"


class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    def update_model_average(self, ma_model, current_model):
        for cur, ma in zip(current_model.parameters(), ma_model.parameters()):
            ma.data = self.update_average(ma.data, cur.data)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1.0 - self.beta) * new


class Queue:
    def __init__(self, max_len: int = 50):
        self.items = []
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def add(self, item):
        self.items.insert(0, item)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, gradnorm_queue: Queue):
    max_grad_norm = 1.5 * gradnorm_queue.mean() + 2.0 * gradnorm_queue.std()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=max_grad_norm, norm_type=2.0
    )
    gn = float(grad_norm)
    if gn > max_grad_norm:
        gradnorm_queue.add(float(max_grad_norm))
        print(f"Clipped gradient {gn:.1f} (allowed {max_grad_norm:.1f})")
    else:
        gradnorm_queue.add(gn)


def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def load_part_pairs(part_i: int):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(f"no part shards; only flat shards in {PAIR_DIR}")
    if not paths:
        raise FileNotFoundError(f"no shards in {PAIR_DIR}")
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    for p in packs:
        if "z_a" not in p or "z_b" not in p:
            raise KeyError(f"{PAIR_DIR} missing z_a/z_b — run the dump cell first")
    waypoints = torch.stack(
        [
            torch.cat([p["z_T"] for p in packs]),
            torch.cat([p["z_a"] for p in packs]),
            torch.cat([p["z_b"] for p in packs]),
            torch.cat([p["x1"] for p in packs]),
        ],
        dim=1,
    )
    n_atoms = torch.cat([p["n_atoms"] for p in packs])
    context = torch.cat([p["context"] for p in packs])
    return waypoints, n_atoms, context


def segment_loss(student, waypoints, node_mask, edge_mask, batch_context):
    """Linear CFM on a random hop. t_clock=(k+u)/3; v* = end-start = dz/du."""
    B = waypoints.shape[0]
    k = torch.randint(0, N_SEG, (B,), device=waypoints.device)
    u = torch.rand(B, 1, device=waypoints.device)
    t = (k.float()[:, None] + u) / N_SEG
    idx = torch.arange(B, device=waypoints.device)
    start, end = waypoints[idx, k], waypoints[idx, k + 1]
    u_b = u[:, None, :]
    xt = (1.0 - u_b) * start + u_b * end
    xt = torch.cat(
        [remove_mean_with_mask(xt[..., :3], node_mask), xt[..., 3:]], -1
    ) * node_mask
    target = (end - start) * node_mask
    pred = student.velocity(xt, t, node_mask, edge_mask, batch_context)
    err = (pred - target) ** 2 * node_mask
    n = node_mask.sum().clamp_min(1)
    loss_x = err[..., :3].sum() / (n * student.n_dims)
    loss_h = err[..., 3:].sum() / (n * student.in_node_nf)
    return loss_x + loss_h


def save_ckpt(path, epoch, avg, best, student, student_ema, opt, gradnorm_queue):
    torch.save(
        {
            "epoch": epoch,
            "avg_loss": avg,
            "best_loss": best,
            "student": student.state_dict(),
            "student_ema": student_ema.state_dict() if student_ema is not None else None,
            "opt": opt.state_dict(),
            "hidden_nf": HIDDEN_NF,
            "lr": LR,
            "batch": BATCH,
            "ema_decay": EMA_DECAY,
            "n_seg": N_SEG,
            "gradnorm_queue": list(gradnorm_queue.items),
        },
        path,
    )


edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
    ),
    in_node_nf=8,
).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
gradnorm_queue = Queue()
gradnorm_queue.add(3000.0)

log(
    f"start 3seg device={device} batch={BATCH} hidden_nf={HIDDEN_NF} lr={LR} "
    f"ema={EMA_DECAY} epochs={EPOCHS} adaptive_clip=True"
)

best_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    waypoints, n_atoms, context = load_part_pairs(part_i)
    perm = torch.randperm(waypoints.shape[0])
    waypoints, n_atoms, context = waypoints[perm], n_atoms[perm], context[perm]
    log(f"epoch={epoch} part={part_i + 1} n={waypoints.shape[0]}")

    student.train()
    running, n_steps = 0.0, 0
    pbar = tqdm(
        range(0, waypoints.shape[0] - BATCH + 1, BATCH),
        desc=f"ep{epoch} part{part_i + 1}",
    )
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = n_atoms[sl].to(device=device, dtype=torch.long)
        node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
        ctx = context[sl].to(device=device, dtype=torch.float32)
        normed = (ctx - norms["mean"]) / norms["mad"]
        batch_context = normed.unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask
        wp = waypoints[sl].to(device=device, dtype=torch.float32)

        loss = segment_loss(student, wp, node_mask, edge_mask, batch_context)
        if not torch.isfinite(loss):
            log(f"WARN non-finite epoch={epoch} step={n_steps}")
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, gradnorm_queue)
        opt.step()
        ema.update_model_average(student_ema, student)

        running += loss.item()
        n_steps += 1
        if n_steps % 20 == 0:
            pbar.set_postfix(loss=f"{running / n_steps:.4f}")

    avg = running / max(n_steps, 1)
    if avg < best_loss:
        best_loss = avg
        save_ckpt(
            CKPT_DIR / f"best_{HIDDEN_NF}_3seg.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt best epoch={epoch} avg_loss={avg:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    log(
        f"epoch={epoch} part={part_i + 1} steps={n_steps} "
        f"avg_loss={avg:.6f} best_loss={best_loss:.6f} "
        f"grad_q_mean={gradnorm_queue.mean():.1f}"
    )

    if (epoch + 1) % CKPT_EVERY == 0:
        save_ckpt(
            CKPT_DIR / f"latest_{HIDDEN_NF}_3seg.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt latest epoch={epoch} avg_loss={avg:.6f}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        log(f"early stop at epoch={epoch} best_loss={best_loss:.6f}")
        break

log(f"done best_loss={best_loss:.6f}")